In [0]:
env = dbutils.widgets.get("environment") if dbutils.widgets.getAll().get("environment") else "dev"
targetTable = f"saleslake_{env}.bronze_{env}.rawInvoice"
srcPath = f"/Volumes/saleslake_{env}/bronze_{env}/vol_saleslake_src_files_{env}/daily_invoice/"

spark.sql(f"""
    COPY INTO {targetTable}
    FROM (
        SELECT
            invoice_id,
            invoice_number,
            customer_id,
            invoice_date,
            due_date,
            subtotal_amount,
            discount_code,
            discount_amount,
            tax_amount,
            total_amount,
            payment_status,
            payment_method,
            payment_date,
            currency,
            region,
            store_id,
            channel,
            created_by,
            current_timestamp() AS ingest_ts
        FROM '{srcPath}'
    )
    FILEFORMAT = CSV
    FORMAT_OPTIONS (
        'header' = 'true'
    )
    COPY_OPTIONS (
        'mergeSchema' = 'true'
    )
    """)

print("Invoice Bronze Load Successful")